In [2]:
import argparse
import logging
from pathlib import Path

from ingestion.parser import parse_pdf
from ingestion.chunker import chunk_document
from ingestion.embedder_indexer import embed_and_index
import config

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger(__name__)

In [3]:
DOCUMENTS = [
    ("pdf", config.DATA_RAW / "ch547_short_term_rental.pdf", "short_term_rental"),
    ("pdf", config.DATA_RAW / "ch591_noise.pdf",             "noise"),
]

In [ ]:
recreate = False # Set to True to recreate the Qdrant collection from scratch
all_chunks = []

for parser_type, source, domain in DOCUMENTS:
    logger.info(f"\n{'='*60}")
    logger.info(f"Processing: {domain}")

    if parser_type == "pdf":
        if not Path(source).exists():
            logger.warning(f"File not found, skipping: {source}")
            continue
        doc = parse_pdf(Path(source), domain)
    else:
        continue  # Unsupported parser types are skipped for now

    chunks = chunk_document(doc)
    all_chunks.extend(chunks)
    logger.info(f"  Total chunks so far: {len(all_chunks)}")

logger.info(f"\n{'='*60}")
logger.info(f"Total chunks to index: {len(all_chunks)}")
embed_and_index(all_chunks, recreate=recreate)